In [ ]:
# dataset github link:
# 
# https://github.com/ad-freiburg/elevant/blob/master/benchmarks/aida-conll-dev.benchmark.jsonl

In [ ]:

import json
import pandas as pd

rows=[json.loads(l) for l in open('aida-conll-dev.benchmark.jsonl')]

df = pd.DataFrame(rows)


types = set(df["type"])
for i in df.iloc[0]["labels"]:  # first row of the dataframe
    print(i)


df.iloc[0]["type"]

{'id': 0, 'span': [10, 24], 'entity_id': 'Q3229147', 'name': 'Leicestershire County Cricket Club', 'type': 'Q43229'}
{'id': 1, 'span': [66, 72], 'entity_id': 'Q84', 'name': 'London', 'type': 'Q27096213|Q43229'}
{'id': 2, 'span': [84, 95], 'entity_id': 'Q912881', 'name': 'West Indies cricket team', 'type': 'Q43229'}
{'id': 3, 'span': [108, 120], 'entity_id': 'Q6827914', 'name': 'Phil Simmons', 'type': 'Q215627'}
{'id': 4, 'span': [151, 165], 'entity_id': 'Q3229147', 'name': 'Leicestershire County Cricket Club', 'type': 'Q43229'}
{'id': 5, 'span': [171, 179], 'entity_id': 'Q3489807', 'name': 'Somerset County Cricket Club', 'type': 'Q43229'}
{'id': 6, 'span': [336, 341], 'entity_id': 'Q3058749', 'name': 'Essex County Cricket Club', 'type': 'Q43229'}
{'id': 7, 'span': [344, 354], 'entity_id': 'Q1788018', 'name': 'Derbyshire County Cricket Club', 'type': 'Q43229'}
{'id': 8, 'span': [359, 365], 'entity_id': 'Q2725632', 'name': 'Surrey County Cricket Club', 'type': 'Q43229'}
{'id': 9, 'span':

KeyError: 'type'

In [ ]:
# Flatten: one row per entity mention
flat_rows = []
for _, row in df.iterrows():
    for lab in row["labels"]:
        start, end = lab["span"]
        flat_rows.append({
            "document_id": row["id"],
            "mention": row["text"][start:end],
            "start": start,
            "end": end,
            "entity_id": lab["entity_id"],
            "entity_name": lab["name"],
            "entity_types": lab["type"],
            "text": row["text"],  # raw document text (the mention's context)
        })

flat_df = pd.DataFrame(flat_rows)
print(flat_df.shape)
flat_df.head(10)

In [ ]:
import re
import time
from pathlib import Path

import requests

WIKIDATA_API = "https://www.wikidata.org/w/api.php"
CACHE_FILE = Path("qid_status_cache.json")

session = requests.Session()
session.headers["User-Agent"] = "qid-validator/0.1 (educational; itisha)"


def _fetch_batch(batch: list[str], max_retries: int = 5) -> dict:
    """One wbgetentities call with retry/backoff on 429."""
    for attempt in range(max_retries):
        resp = session.get(WIKIDATA_API, params={
            "action": "wbgetentities",
            "ids": "|".join(batch),
            "props": "labels|descriptions",
            "languages": "en",
            "languagefallback": 1,  # items whose label lives under "mul" still resolve
            "format": "json",
        }, timeout=30)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 2 ** attempt * 5))
            print(f"rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()["entities"]
    raise RuntimeError(f"still rate-limited after {max_retries} retries")


def check_qids(qids: list[str]) -> dict[str, dict]:
    # load cache: QIDs checked in a previous run are never re-queried
    cache = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}

    unique = list(dict.fromkeys(str(q) for q in qids))
    # sentinels like <NIL>, <NO_MAPPING>, OTHER are not Wikidata items
    queryable = [q for q in unique if re.fullmatch(r"Q\d+", q)]
    todo = [q for q in queryable if q not in cache]
    print(f"{len(unique)} unique ids, {len(queryable)} queryable, {len(todo)} not yet cached")

    for i in range(0, len(todo), 50):
        batch = todo[i : i + 50]
        entities = _fetch_batch(batch)

        # a redirected qid comes back keyed by its target id, with a redirects.from field
        redirect_from = {e.get("redirects", {}).get("from"): k
                         for k, e in entities.items() if e.get("redirects")}
        for qid in batch:
            final = redirect_from.get(qid, qid)
            ent = entities.get(qid) or entities.get(final)
            if not ent or "missing" in ent:
                cache[qid] = {"valid": False, "label": None, "description": None,
                              "redirects_to": None, "url": None}
            else:
                cache[qid] = {
                    "valid": True,
                    "label": ent.get("labels", {}).get("en", {}).get("value"),
                    "description": ent.get("descriptions", {}).get("en", {}).get("value"),
                    "redirects_to": final if final != qid else None,
                    "url": f"https://www.wikidata.org/wiki/{ent.get('id', qid)}",
                }

        # save progress after every batch, so an interruption loses nothing
        CACHE_FILE.write_text(json.dumps(cache))
        time.sleep(1.5)  # stay under the rate limit

    return {q: cache[q] for q in queryable}

In [ ]:
# validate the gold entity QIDs
status = check_qids(flat_df["entity_id"].tolist())

def _field(col: str):
    return flat_df["entity_id"].map(lambda q: status.get(str(q), {}).get(col))

flat_df["qid_valid"]      = _field("valid").fillna(False).astype(bool)  # sentinels (<NIL>, <NO_MAPPING>) -> False
flat_df["wd_label"]       = _field("label")
flat_df["wd_description"] = _field("description")
flat_df["wd_url"]         = _field("url")

# fetch names for the type QIDs (pipe-separated, e.g. "Q27096213|Q43229")
type_qids = sorted({t for ts in flat_df["entity_types"] for t in str(ts).split("|")})
type_status = check_qids(type_qids)
type_label = {q: s["label"] for q, s in type_status.items()}

flat_df["type_labels"] = flat_df["entity_types"].map(
    lambda ts: "|".join(type_label.get(t) or t for t in str(ts).split("|"))
)

print(flat_df.qid_valid.value_counts())
print(flat_df.loc[~flat_df.qid_valid, "entity_id"].value_counts())
flat_df.head(10)

In [ ]:
# split off mentions with no gold Wikidata link (<NIL> = not in any KB, <NO_MAPPING> = no mapping)
is_linked = flat_df["entity_id"].str.fullmatch(r"Q\d+")

nil_df = flat_df[~is_linked].reset_index(drop=True)   # kept aside for NIL/abstention evaluation
flat_df = flat_df[is_linked].reset_index(drop=True)   # flat_df now has only real QIDs

print(f"removed {len(nil_df)} NIL/no-qid mentions -> flat_df is now {flat_df.shape}")
print(flat_df["entity_id"].str.fullmatch(r"Q\d+").all())  # True: no sentinels left
flat_df.head(10)

In [ ]:
# Ambiguous vs non-ambiguous: a surface form is ambiguous when it maps to
# more than one distinct gold QID across the dataset (context decides which).
surface = flat_df["mention"].str.lower()
qids_per_surface = flat_df.groupby(surface)["entity_id"].nunique()
flat_df["n_gold_qids"] = surface.map(qids_per_surface)
flat_df["ambiguous"] = flat_df["n_gold_qids"] > 1

print(flat_df["ambiguous"].value_counts())

# the most ambiguous surface forms, with the competing gold answers
(flat_df[flat_df.ambiguous]
 .assign(s=surface)
 .groupby("s")
 .agg(n_qids=("entity_id", "nunique"),
      gold_answers=("wd_label", lambda x: " / ".join(sorted(set(x)))))
 .sort_values("n_qids", ascending=False)
 .head(15))

In [ ]:
# Balanced eval set: one row per surface form — all 116 ambiguous forms + 116 non-ambiguous forms
RNG = 42
s = linked_df["mention"].str.lower()
linked_df = linked_df.assign(s=s)

ambig_forms = linked_df.groupby("s")["entity_id"].nunique().loc[lambda x: x > 1].index

# one randomly chosen occurrence of each ambiguous surface form (116 rows)
amb_df = (linked_df[linked_df.s.isin(ambig_forms)]
          .groupby("s", as_index=False).sample(1, random_state=RNG)
          .assign(category="ambiguous"))

# one row per non-ambiguous surface form, sampled down to the same size (116 rows)
clear_df = (linked_df[~linked_df.s.isin(ambig_forms)]
            .drop_duplicates("s")
            .sample(len(amb_df), random_state=RNG)
            .assign(category="non_ambiguous"))

eval_df = pd.concat([amb_df, clear_df], ignore_index=True).drop(columns="s")

eval_df.to_csv("el_eval_balanced.csv", index=False)
print(eval_df.category.value_counts())
eval_df.groupby("category").head(5)[["category", "mention", "entity_id", "wd_label"]]